# A3 04 — Sensitivity analyses

This notebook tests whether the main conclusions depend on three analytical choices: excluding the low-visibility 2024 collection, retaining rare matched taxa, and interpreting bone diversity only above a minimum specimen count. It repeats only central effect sizes and produces no duplicate plots.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))
from a3_common import *
processed_dir, output_dir = ensure_a3_directories(ROOT)
aerial = pd.read_pickle(processed_dir / 'aerial_primary.pkl')
bones_primary = pd.read_pickle(processed_dir / 'bones_primary_excluding_2024.pkl')
bones_all = pd.read_pickle(processed_dir / 'bones_including_2024.pkl')
decisions = pd.read_csv(processed_dir / 'taxon_decisions.csv')
shared_taxa = decisions.loc[decisions['matched_analysis'], 'taxon'].tolist()


## Sensitivity 1 — Include the 2024 bone collection

The primary analysis excludes bones collected in 2024 because tall grass reduced visibility. This analysis adds them back and recalculates the main fidelity effect sizes. Estimated death year remains the ecological time variable.

**Plain-language example.** If a poorly lit search finds fewer objects, including it may change the apparent composition even when the underlying population did not change. A robust conclusion should not reverse when those observations are added.


In [2]:
live_all = aerial[aerial['Species'].isin(shared_taxa)].groupby('Species')['Total'].sum()
rows = []
for scenario, bone_df in [('exclude 2024', bones_primary), ('include 2024', bones_all)]:
    bone_all = bone_df[bone_df['Species'].isin(shared_taxa)].groupby('Species')['Total'].sum()
    rows.append({'scenario': scenario, 'comparison': 'OPC overall', **composition_comparison(live_all, bone_all)})
    live_sector = aggregate_composition(aerial, shared_taxa, ['Sector'])
    bone_sector = aggregate_composition(bone_df, shared_taxa, ['Sector'])
    for sector in SECTORS:
        rows.append({'scenario': scenario, 'comparison': sector, **composition_comparison(live_sector.loc[sector], bone_sector.loc[sector])})
sensitivity_2024 = pd.DataFrame(rows)
sensitivity_2024.to_csv(output_dir / 'A3_04_2024_sensitivity.csv', index=False)
sensitivity_2024.round(3)


,scenario,comparison,bray_curtis,pearson,spearman,taxa_compared
0,exclude 2024,OPC overall,0.358,0.789,0.763,15
1,exclude 2024,Eastern,0.390,0.631,0.828,15
2,exclude 2024,Western,0.378,0.821,0.668,15
3,include 2024,OPC overall,0.349,0.798,0.763,15
4,include 2024,Eastern,0.379,0.652,0.847,15
5,include 2024,Western,0.374,0.823,0.668,15


### Interpretation

Including the low-visibility 2024 collection changed overall Bray–Curtis from 0.358 to 0.349 and Pearson r from 0.789 to 0.798. Sector-specific Bray–Curtis changed by 0.011 in the East and 0.004 in the West, without reversing the relative sector pattern. The main fidelity conclusion is therefore insensitive to this exclusion, although the known collection problem still justifies the primary choice.


## Sensitivity 2 — Rare matched taxa

The primary analysis retains every eligible matched taxon. Here, taxa occurring in fewer than 20% of primary bone sector-years are removed, and the overall and sector comparisons are repeated. The retained set is determined once from primary bones and then applied to both datasets.

**Plain-language example.** A taxon observed once may have little influence or may create an unstable comparison. Repeating the analysis without very rare taxa shows whether the conclusion depends on those isolated observations.


In [3]:
bone_presence = (annual_matrix(bones_primary, shared_taxa) > 0).mean(axis=0)
common_taxa = bone_presence[bone_presence >= 0.20].index.tolist()
rare_rows = []
for label, taxa in [('all matched taxa', shared_taxa), ('20% bone occurrence', common_taxa)]:
    live = aerial[aerial['Species'].isin(taxa)].groupby('Species')['Total'].sum()
    bone = bones_primary[bones_primary['Species'].isin(taxa)].groupby('Species')['Total'].sum()
    rare_rows.append({'taxon_rule': label, **composition_comparison(live, bone)})
rare_sensitivity = pd.DataFrame(rare_rows)
rare_sensitivity.insert(1, 'taxa_retained', [len(shared_taxa), len(common_taxa)])
rare_sensitivity.to_csv(output_dir / 'A3_04_rare_taxon_sensitivity.csv', index=False)
print('Taxa retained at 20%:', common_taxa)
rare_sensitivity.round(3)


Taxa retained at 20%: ['Aepyceros melampus', 'Alcelaphus buselaphus', 'Equus burchellii', 'Eudorcas thomsonii', 'Giraffa camelopardalis', 'Kobus ellipsiprymnus', 'Nanger granti', 'Phacochoerus africanus', 'Syncerus caffer', 'Taurotragus oryx']


,taxon_rule,taxa_retained,bray_curtis,pearson,spearman,taxa_compared
0,all matched taxa,15,0.358,0.789,0.763,15
1,20% bone occurrence,10,0.351,0.749,0.539,10


### Interpretation

The 20% rule retained 10 of 15 taxa. Overall Bray–Curtis was nearly unchanged (0.358 versus 0.351), but Pearson r declined from 0.789 to 0.749 and Spearman rho from 0.763 to 0.539. Thus the magnitude-based mismatch is robust, whereas rank agreement remains somewhat sensitive to the arbitrary rare-taxon filter. The complete eligible matched set remains primary.


## Sensitivity 3 — Minimum bone sample size for diversity

For each sector × estimated-death-year sample, annual bone diversity is flagged under minimum summed matched-taxon MNI thresholds of 5, 10, and 15. MNI is first calculated by analytical category within each standardized transect and then summed across contributing, spatially separated transects. The diversity values do not change; only which sector-years are considered interpretable changes. The primary ≥10 rule means summed MNI of 10 or more across eligible matched taxa in that one sector-year. It does not mean 10 skeletal elements, represented taxa, transects, or aerial-census animals. We also list every sector-year shared by the aerial and bone series, including aerial counts, summed matched bone MNI, and represented-taxon richness. A separate scope table states exactly which observations support each analysis.

**Plain-language example.** Combining weathering stages can fill several small cups into fewer fuller cups. That improves each estimate, but a comparison still cannot be trusted if one side contains only two cups.


In [4]:
bone_matrix = annual_matrix(bones_primary, shared_taxa)
threshold_rows = []
for threshold in [5, 10, 15]:
    div = annual_diversity(bone_matrix, minimum_count=threshold)
    for sector in SECTORS:
        subset = div[div['sector'].eq(sector)]
        threshold_rows.append({'minimum_MNI': threshold, 'sector': sector,
                               'years_retained': int(subset['interpret'].sum()),
                               'years_available': int(len(subset))})
threshold_sensitivity = pd.DataFrame(threshold_rows)
threshold_sensitivity.to_csv(output_dir / 'A3_04_bone_threshold_sensitivity.csv', index=False)
display(threshold_sensitivity)

bone_MNI = bone_matrix.sum(axis=1).rename('matched_bone_MNI').reset_index()
bone_MNI['period'] = np.where(bone_MNI['year'] < POST_FENCE_YEAR, PRE_FENCE_LABEL, POST_FENCE_LABEL)
period_feasibility = (bone_MNI.groupby(['sector', 'period'])
    .agg(years_available=('year', 'size'),
         years_MNI_ge_10=('matched_bone_MNI', lambda x: int((x >= 10).sum())))
    .reset_index().assign(scope='fence period'))
living_years = set(aerial['Year'].dropna().astype(int))
overlap_feasibility = (bone_MNI[bone_MNI['year'].isin(living_years)]
    .groupby('sector').agg(years_available=('year', 'size'),
                            years_MNI_ge_10=('matched_bone_MNI', lambda x: int((x >= 10).sum())))
    .reset_index().assign(period='all', scope='live-year overlap'))
analysis_feasibility = pd.concat([period_feasibility, overlap_feasibility], ignore_index=True)
analysis_feasibility = analysis_feasibility[['scope', 'sector', 'period', 'years_available', 'years_MNI_ge_10']]
analysis_feasibility.to_csv(output_dir / 'A3_04_analysis_feasibility.csv', index=False)
display(analysis_feasibility)

# Copy-ready audit of sector-years present in both datasets.
living_matrix = annual_matrix(aerial, shared_taxa)
living_summary = pd.DataFrame({
    'living_count': living_matrix.sum(axis=1),
    'living_taxa': (living_matrix > 0).sum(axis=1),
})
bone_summary = pd.DataFrame({
    'bone_MNI': bone_matrix.sum(axis=1),
    'bone_taxa': (bone_matrix > 0).sum(axis=1),
})
overlapping_years = living_summary.join(bone_summary, how='inner').reset_index()
overlapping_years['period'] = np.where(
    overlapping_years['year'] < POST_FENCE_YEAR, PRE_FENCE_LABEL, POST_FENCE_LABEL)
overlapping_years['bone_MNI_ge_10'] = overlapping_years['bone_MNI'] >= 10
overlapping_years = overlapping_years[[
    'sector', 'year', 'period', 'living_count', 'living_taxa',
    'bone_MNI', 'bone_taxa', 'bone_MNI_ge_10']]
overlapping_years.to_csv(output_dir / 'A3_04_overlapping_live_bone_years.csv', index=False)
display(overlapping_years)

analysis_scope = pd.DataFrame([
    {'analysis': 'Overall live–bone fidelity', 'observations_used': 'All primary observations pooled by dataset',
     'year_matching': 'No', 'bone_minimum': 'Not applied'},
    {'analysis': 'Pre/post change correspondence', 'observations_used': 'All observations pooled within each fence period',
     'year_matching': 'No', 'bone_minimum': 'Not applied'},
    {'analysis': 'Annual bone diversity', 'observations_used': 'Individual bone sector-years',
     'year_matching': 'No', 'bone_minimum': 'Summed matched-taxon MNI ≥10 per sector-year'},
    {'analysis': 'Paired annual live–bone test', 'observations_used': 'Overlapping sector-years only',
     'year_matching': 'Required', 'bone_minimum': 'Not performed: inadequate replication'},
])
analysis_scope.to_csv(output_dir / 'A3_04_analysis_data_scope.csv', index=False)
analysis_scope


,minimum_MNI,sector,years_retained,years_available
0,5,Eastern,12,14
1,5,Western,7,11
2,10,Eastern,11,14
3,10,Western,6,11
4,15,Eastern,7,14
5,15,Western,6,11


,scope,sector,period,years_available,years_MNI_ge_10
0,fence period,Eastern,post-removal (2007+),4,3
1,fence period,Eastern,pre-removal,10,8
2,fence period,Western,post-removal (2007+),4,2
3,fence period,Western,pre-removal,7,4
4,live-year overlap,Eastern,all,7,6
5,live-year overlap,Western,all,7,3


,sector,year,period,living_count,living_taxa,bone_MNI,bone_taxa,bone_MNI_ge_10
0,Eastern,2002,pre-removal,935.0,11,23.0,5,True
1,Eastern,2003,pre-removal,979.0,10,10.0,5,True
2,Eastern,2006,pre-removal,1671.0,14,13.0,6,True
3,Eastern,2008,post-removal (2007+),3965.0,14,11.0,3,True
4,Eastern,2010,post-removal (2007+),2417.0,13,5.0,3,False
5,Eastern,2013,post-removal (2007+),2669.0,14,22.0,7,True
6,Eastern,2014,post-removal (2007+),3506.0,14,23.0,6,True
7,Western,2006,pre-removal,4556.0,11,4.0,3,False
8,Western,2008,post-removal (2007+),5944.0,11,4.0,3,False
9,Western,2010,post-removal (2007+),6625.0,12,1.0,1,False


,analysis,observations_used,year_matching,bone_minimum
0,Overall live–bone fidelity,All primary observations pooled by dataset,No,Not applied
1,Pre/post change correspondence,All observations pooled within each fence period,No,Not applied
2,Annual bone diversity,Individual bone sector-years,No,Summed matched-taxon MNI ≥10 per sector-year
3,Paired annual live–bone test,Overlapping sector-years only,Required,Not performed: inadequate replication


## Sensitivity conclusion

Including the 2024 collection changed overall Bray–Curtis only from 0.358 to 0.349. The 20% occurrence filter retained 10 of 15 taxa and changed Bray–Curtis to 0.351, although Spearman rho fell from 0.763 to 0.539. At the primary threshold of summed matched-taxon MNI ≥10 within one sector × estimated-death-year sample, 11 of 14 Eastern and 6 of 11 Western years were retained; MNI thresholds of 5 and 15 retained 12/7 and 7/6 Eastern/Western years. Adding the early Sweetwaters aerial totals expanded the overlap audit to seven Eastern years, six of which reached matched-taxon MNI ≥10; only two of five Western overlapping years did so, and no Western pre-removal overlap reached the threshold. The table remains an availability audit rather than the input to a performed paired test. Overall fidelity pools all primary observations, period change pools observations within periods, and annual bone diversity alone applies the summed matched-taxon MNI ≥10 sector-year criterion.
